# RAG + Agents: Intelligent Document Retrieval

### What You'll Learn

Building on your knowledge from **3.LangChainRAG** and **4.BuildingAgents**, this notebook shows how to combine the power of **document retrieval** with **intelligent agents**. You'll create agents that can both search the web for current information AND retrieve from your document knowledge base.

**Learning Progression:**
1. **Review: From Basic Agents to RAG Agents** - Why combine them?
2. **Create a Document Retriever Tool** - Turn your RAG system into an agent tool
3. **Multi-Tool Agent** - Combine web search + document retrieval
4. **Intelligent Tool Selection** - Watch agents choose the right tool
5. **Complex Queries** - Handle questions requiring multiple knowledge sources

### Why Combine RAG with Agents?

**RAG Alone:**
- ✅ Great for questions about your documents
- 🚫 Limited to your knowledge base
- ? No access to current information

**Agents Alone:**
- ✅ Can search the web for current info
- ✅ Can perform calculations
- 🚫 Don't know about your specific documents

**RAG + Agents = 🚀 Supercharged AI**
- ✅ **Best of both worlds** - Your documents + web search
- ✅ **Intelligent routing** - Agent chooses the right knowledge source
- ✅ **Current + Historical** - Real-time info + your knowledge base
- ✅ **Complex reasoning** - Multi-step queries across different sources

### Real-World Examples

- **"What's the latest news about our product roadmap?"** → Web search + internal docs
- **"How does our pricing compare to competitors today?"** → Internal pricing docs + current competitor research  
- **"What's our policy on remote work, and how does it compare to industry trends?"** → HR documents + current industry analysis

Let's build this intelligent system! 🤖

## Step 1: Environment Setup

We'll build on the foundations from previous notebooks. Make sure you have the same environment setup:

In [ ]:
import os
import dotenv
from langchain_openai import AzureChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent

dotenv.load_dotenv()

# Reuse Azure OpenAI setup from previous notebooks
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

print("✓ Connected to Azure OpenAI")
print("✓ Ready to build RAG + Agent system")

## Step 2: Create the RAG System (Retriever Tool)

First, let's recreate the RAG system from **3.LangChainRAG** but package it as a **tool** that agents can use.

![rag_retriever_flow.png](../../Assets/images/rag_retriever_flow.png)

### Why Turn RAG into a Tool?

Instead of a standalone RAG system, we create a **retriever tool** that:
- ✅ Can be combined with other tools (web search, calculator, etc.)
- ✅ Let agents decide WHEN to use document retrieval
- ✅ Enables multi-step reasoning across different knowledge sources

### 🔧 Azure OpenAI Configuration

**Important:** Make sure your `.env` file includes the embedding model deployment:
```
AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME=text-embedding-ada-002
```

This notebook uses **Azure OpenAI embeddings** for better enterprise integration and compliance.

In [ ]:
# Step 2.1: Set up document loading and indexing (same as RAG tutorial)
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import AzureOpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load documents (using LangSmith docs as example)
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

# Create embeddings using Azure OpenAI
embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_ADA_DEPLOYMENT"],
    openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

# Create vector store with Azure embeddings
vector_store = FAISS.from_documents(documents, embeddings)

print(f"✓ Loaded and indexed {len(documents)} document chunks")
print(f"✓ Using Azure OpenAI embeddings: {os.environ['AZURE_OPENAI_ADA_DEPLOYMENT']}")
print("✓ Vector store ready for retrieval")

### Step 2.2: Create the Retriever Tool

Now we'll convert our vector store into a **retriever tool** that agents can use. This allows the agent to automatically search our document knowledge base when needed.

In [ ]:
# Step 2.2: Create the retriever tool
# from langchain.tools.retriever import create_retriever_tool

try:
    from langchain.tools.retriever import create_retriever_tool   # newer docs path
except ImportError:
    from langchain_core.tools.retriever import create_retriever_tool  # alt path

# Convert the vector store to a retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Create a retriever tool for agents
retriever_tool = create_retriever_tool(
    retriever,
    "knowledge_base_search", 
    "Search the LangSmith documentation knowledge base. Use this for questions about LangSmith features, setup, usage, and best practices."
)

print("✓ Created retriever tool:")
print(f"  - Name: {retriever_tool.name}")
print(f"  - Description: {retriever_tool.description}")

### Step 2.3: Test the Retriever Tool

Let's test our retriever tool directly to ensure it works:

In [ ]:
# Test the retriever tool directly
test_result = retriever_tool.invoke("What is LangSmith?")
print("Retriever Tool Results:")
print(test_result)

## Step 3: Set Up Web Search Tool

Now let's add the web search capability we learned in **4.BuildingAgents**:

In [ ]:
# Create web search tool (from previous notebook)
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY") or "your-tavily-api-key"

web_search_tool = TavilySearchResults(
    max_results=3,
    search_depth="basic", 
    name="web_search",
    description="Search the web for current information, news, and real-time data. Use this for questions about recent events, current prices, weather, etc."
)

print("✓ Web search tool configured")
print(f"✓ Tool name: {web_search_tool.name}")
print(f"✓ Tool description: {web_search_tool.description}")

## Step 4: Create the Multi-Tool RAG Agent

Now for the exciting part! We'll create an agent that has access to BOTH knowledge sources:

### Agent Architecture

```
User Question
     ↓
Agent Reasoning: "Should I search documents or the web?"
     ↓
Tool Selection:
├── knowledge_base_search (for LangSmith questions)  
└── web_search (for current information)
     ↓
Execute Tool → Get Results → Generate Response
```

In [ ]:
# Combine both tools
tools = [retriever_tool, web_search_tool]

# Create the multi-tool agent
rag_agent = create_react_agent(llm, tools)

print("✓ Created RAG + Web Search Agent with tools:")
for tool in tools:
    print(f"  - {tool.name}: {tool.description}")

## Step 5: Test Intelligent Tool Selection

Let's test the agent's ability to choose the right tool for different types of questions:

### Test 1: Document-Specific Question
This should use the **knowledge base search** tool:

In [ ]:
# Test 1: Question about LangSmith (should use retriever tool)
response = rag_agent.invoke({
    "messages": [("human", "What is LangSmith and how do I get started?")]
})

print("=== Test 1: LangSmith Question ===")
print("Agent Response:")
print(response["messages"][-1].content)
print("\n" + "="*50 + "\n")

### Test 2: Current Information Question
This should use the **web search** tool:

In [ ]:
# Test 2: Question about current events (should use web search)
response = rag_agent.invoke({
    "messages": [("human", "What's the current weather in San Francisco?")]
})

print("=== Test 2: Current Weather Question ===")
print("Agent Response:")
print(response["messages"][-1].content)
print("\n" + "="*50 + "\n")

### Test 3: Complex Question Requiring Both Tools
This should use **BOTH** tools to provide a comprehensive answer:

In [ ]:
# Test 3: Complex question requiring both knowledge sources
response = rag_agent.invoke({
    "messages": [("human", "How do I set up LangSmith monitoring, and what are the current best practices for LLM observability in 2024?")]
})

print("=== Test 3: Complex Question (LangSmith + Current Best Practices) ===")
print("Agent Response:")
print(response["messages"][-1].content)
print("\n" + "="*50 + "\n")

---

## Congratulations! 🎉

You've successfully built a **RAG + Agent Integration System**!

### What You've Accomplished

Let's recap this incredible journey:

1. **Document Knowledge Base** ✓
   - Loaded and processed LangSmith documentation
   - Created vector embeddings for semantic search
   - Built a retriever tool for document queries

2. **Multi-Tool Agent** ✓
   - Combined document search with web search
   - Created intelligent tool selection
   - Implemented autonomous reasoning

3. **Intelligent Tool Routing** ✓
   - Agent chooses retriever for LangSmith questions
   - Agent chooses web search for current information
   - Handles complex queries requiring multiple tools

### Key Benefits of RAG + Agents

**Why This Matters:**
- ✅ **Best of Both Worlds** - Combines reliable document knowledge with real-time information
- ✅ **Intelligent Decision Making** - Agent chooses the right information source automatically
- ✅ **Extensible Architecture** - Easy to add new knowledge bases or tools
- ✅ **Production Ready** - Scalable pattern for enterprise applications

### Real-World Applications

This pattern enables:
- **Customer Support Bots** - Company docs + live troubleshooting
- **Research Assistants** - Academic papers + current news
- **Developer Tools** - Code documentation + Stack Overflow search
- **Business Intelligence** - Internal reports + market data

### Next Steps

**📚 Continue Learning:**
- **Next Notebook: `6.AdvancedAgents.ipynb`** - Multi-agent systems and orchestration
- **Production Deployment** - Deploy with FastAPI or LangServe
- **Advanced Memory** - Persistent conversation history
- **Custom Tools** - Build domain-specific capabilities

**But wait... let's take this one step further! 🚀**

---

## 🌟 Bonus Section: Creating a User Interface with Gradio

**From Technical Demo to User-Friendly Application!**

Let's transform our powerful RAG + Agent system into a web-based chat interface that anyone can use. Instead of running code cells, users will simply visit a webpage and chat with our AI assistant!

### Why Add a UI?

- ✅ **Democratize Access** - Non-technical users can benefit from your AI
- ✅ **Real-World Application** - Turn research into a production-ready tool
- ✅ **Easy Sharing** - Share your AI assistant with colleagues instantly
- ✅ **Professional Presentation** - Impress stakeholders with a polished interface

### Step 1: Install Gradio

First, let's install the Gradio library for creating web interfaces:

In [ ]:
# Install Gradio for web interface
# !pip install gradio

import gradio as gr
print("✓ Gradio installed and imported")

### Step 2: Create the Chat Function

Now we'll create a function that wraps our agent for the chat interface:

In [ ]:
def chat_with_agent(message, history):
    """
    Chat function that integrates with our RAG + Agent system
    
    Args:
        message: User's input message
        history: Chat history (handled by Gradio)
    
    Returns:
        Agent's response
    """
    try:
        # Invoke our agent with the user's message
        result = rag_agent.invoke({
            "messages": [("human", message)]
        })
        
        # Extract the response from the agent's output
        response = result["messages"][-1].content
        return response
        
    except Exception as e:
        return f"Sorry, I encountered an error: {str(e)}"

print("✓ Chat function created")
print("✓ Ready to connect agent to web interface")

### Step 3: Simple Chat Interface

Let's start with a basic chat interface:

In [ ]:
# Simple Chat Interface
simple_interface = gr.ChatInterface(
    chat_with_agent,
    title="🤖 RAG + Agent Assistant",
    description="Ask me about LangSmith or current information!",
    examples=[
        "What is LangSmith?",
        "How can LangSmith help with testing?", 
        "What's the weather like in San Francisco?",
        "Tell me about recent AI news"
    ]
)

# Launch the interface (uncomment to run)
# simple_interface.launch(share=True)

print("✓ Simple chat interface ready!")
print("✓ Uncomment the launch line to start the web interface")

### Step 4: Enhanced Chat Interface

Now let's create a more polished, professional interface:

In [ ]:
# Enhanced Chat Interface with Custom Styling
enhanced_interface = gr.ChatInterface(
    chat_with_agent,
    chatbot=gr.Chatbot(
        height=400,
        avatar_images=["👤", "🤖"]  # User and bot avatars
    ),
    textbox=gr.Textbox(
        placeholder="Hi! I'm your AI assistant. Ask me about LangSmith or any current information...", 
        container=False, 
        scale=7
    ),
    title="🚀 Intelligent RAG + Agent System",
    description="Powered by LangChain • Combines Document Knowledge + Real-time Web Search",
    theme="soft",
    examples=[
        "What is LangSmith and how do I get started?",
        "How can LangSmith help with testing AI applications?",
        "What's the current weather in San Francisco?",
        "Tell me about recent developments in AI",
        "Compare LangSmith with other LLM tools"
    ]
)

# Launch the enhanced interface - try with simpler parameters first
try:
    # Start with basic launch - this usually works in most environments
    enhanced_interface.launch(share=True)
    print("✅ Enhanced chat interface launched successfully!")
    print("✅ Check the output above for the local and public URLs")
except Exception as e:
    print(f"❌ Error launching with share=True: {e}")
    print("⚠️  Trying local launch only...")
    try:
        # Fallback to local-only launch
        enhanced_interface.launch(share=False)
        print("✅ Enhanced chat interface launched locally!")
        print("✅ Access it at: http://127.0.0.1:7860")
    except Exception as e2:
        print(f"❌ Error with local launch: {e2}")
        print("💡 Try running the simple interface instead (previous cell)")

print("✓ Features: Custom avatars, professional theme, example prompts")
print("✓ Your RAG + Agent system is now available as a web application!")

### 🎯 What Just Happened?

**From Code to Web App in Minutes!**

We just transformed our sophisticated RAG + Agent system into a user-friendly web application! Here's the magic:

#### The Architecture:
```
User Browser → Gradio Interface → chat_with_agent() → RAG Agent → Tools → Response
```

#### Key Components:
1. **Gradio ChatInterface** - Handles the web UI, chat history, and user interactions
2. **chat_with_agent()** - Wrapper function that connects Gradio to our agent
3. **RAG Agent** - Our existing multi-tool agent (unchanged!)
4. **Tools** - Document retriever + web search (still working behind the scenes)

#### The User Experience:
- ✅ **No Code Required** - Users just type and chat
- ✅ **Real-time Responses** - See the agent thinking and responding
- ✅ **Professional Interface** - Clean, modern web design
- ✅ **Shareable Link** - `share=True` creates a public URL
- ✅ **Mobile Friendly** - Works on phones and tablets

### 🚀 Ready to Launch Your AI Assistant?

**To start the web interface:**

1. **Uncomment** either launch line in the cells above
2. **Run** the cell 
3. **Click** the Gradio URL that appears
4. **Share** the public link with others (if using `share=True`)

**Example URLs you'll see:**
- Local: `http://127.0.0.1:7860`
- Public: `https://abc123.gradio.live` (shareable worldwide!)

### 💡 Production Deployment Options

**Beyond Gradio for Production:**

1. **FastAPI + LangServe** - Production-grade REST API
2. **Streamlit** - Alternative UI framework  
3. **Docker Containers** - Easy deployment anywhere
4. **Cloud Hosting** - AWS, Azure, GCP integration
5. **Authentication** - Add user login and security

### 🏆 What You've Built

**A Complete AI Application Stack:**
- ✅ **Backend**: RAG + Agent system with multiple tools
- ✅ **Frontend**: Professional web interface
- ✅ **Integration**: Seamless user experience
- ✅ **Deployment**: Ready for production use